In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
from llama_index.embeddings.openai import OpenAIEmbedding

# Load environment variables
load_dotenv(find_dotenv())  
embed_model = OpenAIEmbedding(model_name = os.environ.get("EMBED_MODEL"))

if embed_model is None:
    raise ValueError("EMBED_MODEL environment variable is not set!")

# Load documents
documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()



In [3]:
# merge into a single large document rather than the one document per page

from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [4]:
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[4096, 1024, 256])

In [5]:
nodes = node_parser.get_nodes_from_documents([document])

In [6]:
len(nodes)

803

### build auto_merging engine

In [7]:
from llama_index.llms.openai import OpenAI

# llm = OpenAI(model ="gpt-3.5-turbo", temperature=0.1)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [8]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model=embed_model
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = node_parser

In [10]:
from llama_index.core.node_parser import get_leaf_nodes

leaf_nodes = get_leaf_nodes(nodes)
print(leaf_nodes[30].text)

Sadly, many people rejected that gospel; even some of those who accepted it changed gospel doctrines and ordinances and fell into unbelief and apostasy.
- Our Father in Heaven sent His Beloved Son, Jesus Christ, to earth. He performed miracles and taught His gospel. He accomplished the Atonement and was resurrected.
- Beginning with the First Vision, God has again reached out in love to His children. He restored the gospel of Jesus Christ and His priesthood authority and organized His Church on the earth through the Prophet Joseph Smith. The Book of Mormon is convincing evidence of this Restoration.

As you help investigators see the pattern of apostasy and restoration, they will be prepared to understand the great need for the latter-day Restoration. They will see the need to accept the restored gospel, receive the ordinances of salvation by the authority of the restored priesthood, and follow the way to eternal life. Help people recognize that the Church is not just another religion,

In [9]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import load_index_from_storage

if not os.path.exists("../VectorStore"):
    os.makedirs("../VectorStore")
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    automerging_index = VectorStoreIndex(
        leaf_nodes, storage_context=storage_context)

    automerging_index.storage_context.persist(persist_dir="../VectorStore")

else:
    automerging_index = load_index_from_storage(StorageContext.from_defaults(persist_dir="../VectorStore"))

In [11]:
from llama_index.postprocessor.cohere_rerank import CohereRerank
load_dotenv(find_dotenv())
cohere_rerank = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"], 
    top_n=2,
)

In [12]:
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

automerging_retriever = automerging_index.as_retriever(similarity_top_k=6)

retriever = AutoMergingRetriever(
    automerging_retriever,
    automerging_index.storage_context,
    verbose= True
)

auto_merging_engine = RetrieverQueryEngine.from_args(
    retriever, node_postprocessors=[cohere_rerank]
)

### Evaluation

In [13]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()


/tmp/ipykernel_27439/3827208637.py:1: DeprecationWarning: The `trulens_eval` module is deprecated. See https://trulens.org/docs/trulens/guides/trulens_eval_migration for instructions on migrating to `trulens.*` modules.
  from trulens_eval import Tru
/tmp/ipykernel_27439/3827208637.py:3: DeprecationWarning: Class `TruSession` has moved:
	New import: `from trulens.core.session import TruSession`
 See https://trulens.org/docs/trulens/guides/trulens_eval_migration for instructions on migrating to `trulens` modules.
  tru = Tru()


🦑 TruSession initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [14]:
import numpy as np
from trulens.apps.llamaindex import TruLlama
from trulens.core import Feedback
from trulens.providers.openai import OpenAI

# Initialize provider class
provider = OpenAI(model_engine="gpt-4o-mini")

# select context to be used in feedback. the location of context is app specific.

context = TruLlama.select_context(auto_merging_engine)

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()
# Question/statement relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [15]:
from trulens.apps.llamaindex.guardrails import WithFeedbackFilterNodes

# note: feedback function used for guardrail must only return a score, not also reasons
f_context_relevance_score = Feedback(provider.context_relevance)

filtered_query_engine = WithFeedbackFilterNodes(
    auto_merging_engine, feedback=f_context_relevance_score, threshold=0.5
)

In [16]:
tru_query_engine_recorder = TruLlama(
    auto_merging_engine,
    app_name="LlamaIndex_App",
    app_version="02_Advanced_Retriever",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

🦑 TruSession initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [17]:
import pandas as pd

# Read the CSV file and drop the 'Unnamed: 0' column
df = pd.read_csv("../dataset_eval/20_dataset.csv").drop(columns=['Unnamed: 0'])

# Extract the first 20 questions into a list
questions = df['question'][:20].tolist()

# Print the list of questions
print(questions)


['What strategies can be used to make a message easy to understand when teaching?', 'What should teachers do with unfamiliar words to ensure their message is easy to understand?', 'What are some effective study techniques to enhance understanding and retention of material?', 'What is the purpose of using a study journal in your scripture study?', 'What is the importance of organizing and summarizing lesson plans for effective teaching?', 'What is the recommended method for highlighting key words when marking scriptures?', 'What is the significance of beginning study activities with a prayer?', 'What is the significance of the power of ordination in the context of missionary work?', 'What is the purpose of marking scriptures in relation to applying gospel teachings?', 'What is the significance of preaching the gospel according to President Lorenzo Snow?', 'What is the relationship between the Atonement and missionary work according to President Howard W. Hunter?', 'What are the blessing

In [18]:
# evaluate
for question in questions:
    with tru_query_engine_recorder as recording:
        response = auto_merging_engine.query(question)

> Merging 3 nodes into parent node.
> Parent node id: 63b7ba91-684f-4b9b-b13f-7004075fbda2.
> Parent node text: Use Study Resources (Page 37)
- Use the study aids in the LDS edition of the scriptures (Topical ...

> Merging 3 nodes into parent node.
> Parent node id: 4a11252f-bcc1-40a8-8ff2-ff1081645219.
> Parent node text: Marking Scriptures (Page 38)

Marking your scriptures can assist you in thinking deeply about a p...

> Merging 1 nodes into parent node.
> Parent node id: a0ea3ebd-aa2b-4849-8414-fe3420d454c7.
> Parent node text: Cultural Views of Scriptures (Page 195)
Most religions have sacred texts or books of scripture, b...

> Merging 3 nodes into parent node.
> Parent node id: 4a11252f-bcc1-40a8-8ff2-ff1081645219.
> Parent node text: Marking Scriptures (Page 38)

Marking your scriptures can assist you in thinking deeply about a p...



In [19]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
# records.head()

In [20]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Groundedness,Context Relevance,Answer Relevance
0,"""What strategies can be used to make a message easy to understand when teaching?""","""To make a message easy to understand when teaching, consider the following strategies:\n\n1. Define unfamiliar words simply and clearly, ensuring that your audience can grasp the meaning without confusion.\n2. Use a vocabulary that is appropriate for the audience, similar to how a physician adjusts their language based on the setting and audience.\n3. Familiarize yourself with key definitions and terms relevant to your message, utilizing resources like True to the Faith and the Bible Dictionary for additional clarity.\n4. Incorporate a list of key terms at the end of lessons to help learners review and understand important concepts.\n5. Continuously strive to simplify your message throughout the teaching process, making it accessible to all learners.""",0.866667,0.666667,1.000000
1,"""What should teachers do with unfamiliar words to ensure their message is easy to understand?""","""Teachers should learn to define unfamiliar words simply and make the message easy to understand. They should study key definitions and terms provided in the lessons, utilize resources like True to the Faith and the Bible Dictionary for additional definitions, and write these definitions in their study journal.""",1.000000,1.000000,1.000000
2,"""What are some effective study techniques to enhance understanding and retention of material?""","""To enhance understanding and retention of material, consider the following techniques: \n\n1. Prepare your mind and body by exercising, showering, and praying before studying.\n2. Choose a dedicated study space, such as a desk or table, to help you stay organized and alert.\n3. Approach your study by topic, which can help in structuring your understanding.\n4. Engage with the material by asking reflective questions about the author's message and its relevance to your life.\n5. Use visualization techniques to create mental images related to the content, which can deepen comprehension.\n6. Explore additional learning activities provided at the end of each chapter to reinforce your understanding and apply what you've learned creatively.""",0.611111,0.833333,1.000000
3,"""What is the purpose of using a study journal in your scripture study?""","""The purpose of using a study journal in scripture study is to help understand, clarify, and remember what is being learned. It allows for the recording of thoughts and impressions, which can enhance the likelihood of receiving further insights. Additionally, it serves as a tool to recall spiritual experiences, recognize personal growth, and organize key information for future reference. Regular use of the journal during various study sessions and meetings aids in applying the learned material effectively.""",0.916667,1.000000,1.000000
4,"""What is the importance of organizing and summarizing lesson plans for effective teaching?""","""Organizing and summarizing lesson plans is crucial for effective teaching as it helps ensure that the doctrine is presented clearly and with spiritual power. A well-structured lesson plan allows the teacher to focus on key elements such as doctrine, baptismal interview questions, and commitments. Additionally, incorporating scriptures, questions, and examples enhances the teaching experience and can be tailored to fit the duration of the teaching appointments. This preparation is essential at any stage of the teaching process to facilitate understanding and engagement.""",0.888889,0.666667,1.000000
5,"""What is the recommended method for highlighting key words when marking scriptures?""","""The recommended method for highlighting key words when marking scriptures includes underlining only a few key words to emphasize the verse, section, or chapter. Additionally, you can circle or underline key words and use straight lines to link closely related words. This approach helps maintain clarity and f

In [21]:
tru.get_leaderboard()

,,Groundedness,Context Relevance,Answer Relevance,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,02_Advanced_Retriever,0.891944,0.783333,0.916667,13.7,0.000103
